## Installing Libraries

In [1]:
!pip install pymupdf 
!pip install nltk 
!pip install tqdm
!pip install sentence-transformers
!pip install scikit-learn
!install gradio


[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
'install' is not recognized as an internal or external command,
operable program or batch file.


## Code

In [ ]:
import gradio as gr
import fitz 
import re
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline

# -------------------
# PDF Parsing & Chunking
# -------------------
def extract_text_from_pdf(path):
    doc = fitz.open(path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text.strip()

def regex_sent_tokenize(text):
    sentence_endings = re.compile(r'(?<=[.!?])\s+(?=[A-Z])')
    return sentence_endings.split(text)

def chunk_text(text, max_tokens=300):
    sentences = regex_sent_tokenize(text)
    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        token_count = len(sentence.split())
        if current_length + token_count > max_tokens:
            chunks.append(" ".join(current_chunk))
            current_chunk = [sentence]
            current_length = token_count
        else:
            current_chunk.append(sentence)
            current_length += token_count

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# -------------------
# Embedding & Retrieval
# -------------------
embedder = SentenceTransformer("all-MiniLM-L6-v2")
generator = pipeline("text2text-generation", model="google/flan-t5-small")

def embed_chunks(chunks):
    embeddings = embedder.encode(chunks, show_progress_bar=False)
    return [{"text": chunk, "embedding": emb} for chunk, emb in zip(chunks, embeddings)]

def embed_query(query):
    return embedder.encode([query])[0]

def retrieve_top_k_chunks(query, chunk_data, top_k=3):
    query_vec = embed_query(query)
    chunk_vectors = np.array([item["embedding"] for item in chunk_data])
    scores = cosine_similarity([query_vec], chunk_vectors)[0]
    top_indices = scores.argsort()[::-1][:top_k]
    return [chunk_data[i]["text"] for i in top_indices]

def generate_answer(query, retrieved_chunks):
    context = "\n".join(retrieved_chunks)
    prompt = f"Context:\n{context}\n\nQuestion:\n{query}"
    result = generator(prompt, max_length=200, do_sample=False)[0]["generated_text"]
    return result.strip()

# -------------------
# Gradio Interface
# -------------------
document_chunks = []
document_embeddings = []

def load_document(file):
    global document_chunks, document_embeddings
    text = extract_text_from_pdf(file.name)
    chunks = chunk_text(text)
    document_embeddings = embed_chunks(chunks)
    document_chunks = chunks
    return f"Document loaded with {len(chunks)} chunks."

def answer_question(query):
    if not document_embeddings:
        return "Please upload a document first."
    top_chunks = retrieve_top_k_chunks(query, document_embeddings)
    return generate_answer(query, top_chunks)

with gr.Blocks() as demo:
    gr.Markdown("# Document QA Chatbot")
    gr.Markdown("Upload a PDF and ask questions about its content.")

    with gr.Row():
        pdf_input = gr.File(label="Upload PDF", type="filepath")
        upload_output = gr.Textbox(label="Status", interactive=False)

    with gr.Row():
        question_input = gr.Textbox(label="Ask a question about the document...")
        answer_output = gr.Textbox(label="Answer", interactive=False)

    pdf_input.change(fn=load_document, inputs=pdf_input, outputs=upload_output)
    question_input.submit(fn=answer_question, inputs=question_input, outputs=answer_output)

demo.launch()


c:\Users\chandresh.kanyal\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use cpu


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
